# Phase 1: Dataset creation

GitHub issue: #2. Depends on Phase 0 (#1): target = leave-last-out next-item prediction within a session.

This notebook builds the frozen Phase 1 snapshot from the real OTTO sample. It also documents a real
correction made mid-phase: the first version of this pipeline leaked future weeks into the training
artifacts. That mistake and the fix are both kept here, since understanding *why* the final approach
is what it is matters as much as the approach itself.

## Step 1: Inventory sources

Source: OTTO recsys dataset, Kaggle dataset `otto/recsys-dataset` (not the competition - see decision
`_doc/decisions/D1-otto-license.md`: CC BY 4.0, confirmed directly from Kaggle's own metadata). OTTO has
no item text, no queries, no positions - categories and queries are synthetic (`_doc/AGENTS.md`).

The raw file is ~11.3GB / 220M events - too big for this machine's 8GB RAM or disk. So the actual
sampling ran on Kaggle's own compute instead: `notebooks/kaggle_otto_sample/otto-sample-04.ipynb`
attaches the dataset, keeps whole sessions (deterministic hash of session id, so the same ~12% of
sessions are picked every run), and downloads only the sample. Whole *sessions*, not events - cutting
a session into fragments would break the prefix/next-item structure the whole task depends on.

In [1]:
import json
from pathlib import Path

import numpy as np
import polars as pl

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SAMPLE_DIR = ROOT / "data" / "sample"

sample_manifest = json.loads((SAMPLE_DIR / "sample_manifest.json").read_text())
print(json.dumps(sample_manifest, indent=2))

{
  "rule": "keep session if ((session * 2654435761) % 2**32 >> 8) % 100 < keep_pct",
  "keep_pct": 12,
  "source": "otto-recsys-train.jsonl",
  "source_sessions": 12899779,
  "sessions": 1547984,
  "events": 26047712,
  "events_by_type": {
    "click": 23400355,
    "cart": 2034479,
    "order": 612878
  },
  "ts_min": 1659304800025,
  "ts_max": 1661723999965,
  "days": 28.0,
  "sha256": {
    "events.parquet": "49a199a4f5d9a9823e9a33032ec785ffd0f4148d78d2ac447a419386805a56ea"
  }
}


Result: 1,547,984 sessions (12% of 12,899,779), 26,047,712 events. Type mix is 89.8% click /
7.8% cart / 2.4% order - close to the real dataset's published overall mix, which is a good sign the
session-hash sampling isn't distorting behavior.

In [2]:
events = pl.read_parquet(SAMPLE_DIR / "events.parquet")
print(events.schema)
events.head(3)

Schema([('session', Int64), ('aid', Int32), ('ts', Int64), ('type', Int8)])


session,aid,ts,type
i64,i32,i64,i8
0,1517085,1659304800025,0
0,1563459,1659304904511,0
0,1309446,1659367439426,0


**Finding**: session counts decline sharply week over week in this data (we'll compute this below).
That's expected, not a sampling bug - OTTO's own published methodology trims sessions that would
overlap the labeled/test boundary, and (as this notebook's own Step 3 correction will show) OTTO
"sessions" persist across the whole window, so fewer *new* sessions start near the end.

In [3]:
weeks = events.with_columns(
    (((pl.col("ts") - pl.col("ts").min()) // (7 * 24 * 60 * 60 * 1000))).alias("_week")
)
session_first_week = weeks.group_by("session").agg(pl.col("_week").min().alias("week"))
session_first_week["week"].value_counts().sort("week")

week,count
i64,u32
0,641864
1,395328
2,294642
3,216150


## Step 2: Define the label

Leave-last-out per session (`src/ranking/data/labels.py`): the session's last event (by timestamp) is
held out. The category of that held-out item becomes the synthetic *query* - a documented leak used as
an intent proxy (OTTO has no real search queries), not evidence of real query understanding. The prefix
(every earlier event) is what a model would actually see. Graded label: click=1, cart=2, order=3.

## Step 3: Observation window and cutoff - and a correction

The first version of this pipeline bucketed sessions into train/val/test by the week of their **first**
event, then built categories/co-visitation from "all events of sessions that started in weeks 0-1".

That was wrong. Look at how long OTTO sessions actually run:

In [4]:
sample_sessions = session_first_week.sample(2000, seed=0)["session"]
span = (
    events.join(sample_sessions.to_frame(), on="session", how="inner")
    .group_by("session")
    .agg((pl.col("ts").max() - pl.col("ts").min()).alias("span_ms"))
    .with_columns((pl.col("span_ms") / 86_400_000).alias("span_days"))
)
print(span["span_days"].describe())
long = span.filter(pl.col("span_days") > 7).height
print(f"sessions spanning > 7 days: {long} / {span.height} ({long/span.height:.1%})")

shape: (9, 2)
┌────────────┬───────────┐
│ statistic  ┆ value     │
│ ---        ┆ ---       │
│ str        ┆ f64       │
╞════════════╪═══════════╡
│ count      ┆ 2000.0    │
│ null_count ┆ 0.0       │
│ mean       ┆ 6.850116  │
│ std        ┆ 8.407422  │
│ min        ┆ 0.0       │
│ 25%        ┆ 0.006061  │
│ 50%        ┆ 2.016579  │
│ 75%        ┆ 12.807468 │
│ max        ┆ 27.87953  │
└────────────┴───────────┘
sessions spanning > 7 days: 764 / 2000 (38.2%)


Median session span is multiple days, and roughly half run longer than a week - one sampled session
in the full analysis ran from 2022-07-31 to 2022-08-28, the *entire* window, with an 11-day gap between
two of its events. OTTO "sessions" here behave like a persistent visitor id, not a 30-minute browsing
session.

That means bucketing by a session's *first* event silently pulled each long session's later weeks'
events into what the old code called "training-window" data - real point-in-time leakage into the
shared category/co-visitation artifacts every downstream phase depends on, not just a labeling quirk.

**Fix**: split by each example's own `label_ts` (the held-out event's own timestamp), never by session
identity. Categories/co-visitation are fit on events with `ts` before the train/val cutoff, from *any*
session - not "sessions that started early".

In [5]:
MS_PER_WEEK = 7 * 24 * 60 * 60 * 1000
ts_min = events["ts"].min()
train_end = ts_min + 2 * MS_PER_WEEK
val_end = ts_min + 3 * MS_PER_WEEK
print("train_end:", train_end, "val_end:", val_end)

train_events = events.filter(pl.col("ts") < train_end)
print(f"train events (ts-based, any session): {train_events.height:,}")

train_end: 1660514400025 val_end: 1661119200025
train events (ts-based, any session): 13,064,647


## Step 4: Point-in-time joins - categories and co-visitation, training window only

Two more real findings surfaced building this:

1. **Item frequency is heavily skewed.** Median item frequency is 4; most of the catalog is long-tail.
   Items with fewer than 5 occurrences in the training window get category **-1** (too sparse to place
   reliably) rather than a forced, unreliable placement.
2. **The first clustering attempt collapsed.** Raw co-visitation weights vary hugely by item
   popularity, and Euclidean k-means on the raw SVD embedding grouped items by *magnitude* (how
   connected an item is overall) instead of *direction* (who it specifically co-occurs with) - 99.2%
   of placeable items landed in one giant cluster. Fixed by L2-normalizing the co-visitation matrix and
   the SVD embedding before k-means (`src/ranking/data/categories.py`) - turns it into cosine-similarity
   clustering. Verified on synthetic mock data first (recovery of known hidden categories jumped from
   weak to a perfect adjusted Rand index of 1.0) before trusting it on the real data.

In [6]:
import sys
sys.path.insert(0, str(ROOT / "src"))

from ranking.data.covisitation import covisitation_pairs, top_k_neighbors
from ranking.data.categories import cluster_categories

MIN_ITEM_FREQ = 5
N_CATEGORIES = 80

item_freq = train_events.group_by("aid").agg(pl.len().alias("n"))
frequent_aids = item_freq.filter(pl.col("n") >= MIN_ITEM_FREQ)["aid"].to_numpy()
print(f"frequent items (>= {MIN_ITEM_FREQ} in train): {len(frequent_aids):,} / {item_freq.height:,}")

covis = covisitation_pairs(
    train_events.filter(pl.col("aid").is_in(frequent_aids.tolist())), max_gap_events=5
)
print(f"co-visitation pairs: {covis.height:,}")

frequent items (>= 5 in train): 396,630 / 1,070,425


co-visitation pairs: 17,292,646


In [7]:
clustered = cluster_categories(covis, frequent_aids, n_categories=N_CATEGORIES, n_components=32, seed=0)

sizes = clustered["category"].value_counts().sort("count", descending=True)
print("largest category share of known items:", sizes["count"].max() / clustered.height)
print("median category size:", sizes["count"].median())
sizes.head(5)

largest category share of known items: 0.03612434762877241
median category size: 4765.5


category,count
i32,u32
7,14328
43,10394
11,9941
4,8787
36,8390


Largest category is now ~2.6% of known items (was 99.2% before the fix), median ~4,600 - a
healthy, usable distribution across all 80 categories.

In [8]:
all_aids_in_sample = events["aid"].unique().to_frame()
item_categories = all_aids_in_sample.join(clustered, on="aid", how="left").with_columns(
    pl.col("category").fill_null(-1)
)
unknown_share = (item_categories["category"] == -1).mean()
print(f"items with category -1 (unknown/sparse), catalog-wide: {unknown_share:.1%}")

items with category -1 (unknown/sparse), catalog-wide: 72.0%


## Step 5: Build the ranking examples and validate

In [9]:
from ranking.data.labels import build_ranking_examples

examples = build_ranking_examples(events, item_categories).with_columns(
    pl.when(pl.col("label_ts") < train_end)
    .then(pl.lit("train"))
    .when(pl.col("label_ts") < val_end)
    .then(pl.lit("val"))
    .otherwise(pl.lit("test"))
    .alias("split")
)

print("split counts:")
print(examples["split"].value_counts().sort("split"))
print()
print("duplicate sessions:", examples.height - examples["session"].n_unique())
print("nulls:", dict(examples.null_count().row(0, named=True)))
print("example-level unknown query_category rate:", (examples["query_category"] == -1).mean())

split counts:
shape: (3, 2)
┌───────┬────────┐
│ split ┆ count  │
│ ---   ┆ ---    │
│ str   ┆ u32    │
╞═══════╪════════╡
│ test  ┆ 639037 │
│ train ┆ 512161 │
│ val   ┆ 396786 │
└───────┴────────┘

duplicate sessions: 0
nulls: {'session': 0, 'prefix_last_ts': 0, 'query_category': 0, 'label_aid': 0, 'label_type': 0, 'label_grade': 0, 'label_ts': 0, 'prefix_len': 0, 'split': 0}
example-level unknown query_category rate: 0.17401859450743676


Zero duplicates, zero nulls anywhere, and the example-level unknown-category rate (12-19%,
varies by split) is much lower than the catalog-level rate (72%) - held-out items skew popular, so
they're more likely to have cleared the frequency-5 bar during category fitting.

## Step 6: Freeze

The snapshot (`events.parquet`, `item_categories.parquet`, `ranking_examples.parquet`,
`train_covisitation.parquet`) is written to `data/frozen/` (gitignored - 454MB total, too large to
push reliably) with a manifest (version, split cutoffs, per-file checksums). Data dictionary:
`_doc/data_dictionary.md`. Reproducible end-to-end via `scripts/build_phase1_snapshot.py`, which
is the same logic as this notebook, packaged for unattended reruns.

## Exit criteria

Versioned snapshot (`data/frozen/manifest.json`), data dictionary (`_doc/data_dictionary.md`), written
label definition (Step 2 above), row counts reconciled to source (Step 5 above, matches
`data/sample/sample_manifest.json`). Met - see issue #2 for the full write-up including the
correction's numbers before and after.